In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.7 Thermodynamic Potentials, Legendre Transforms, and the Maxwell Relations

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.7",
    title="Thermodynamic Potentials, Legendre Transforms, and the Maxwell Relations",
    blurb="The structure of thermodynamics. Entropy and energy have the wrong "
    "natural variables for a laboratory — so we trade variables for their "
    "conjugate slopes using the Legendre transform, the very operation that "
    "took the Lagrangian to the Hamiltonian in Volume II. Out come the free "
    "energies, a web of identities called the Maxwell relations, and the "
    "principle that each potential is minimized at equilibrium.",
    difficulty="advanced",
    estimate="200–240 min",
)

## Notebook overview

[§5.6](ideal-gas-fundamental-relation.ipynb) gave us the fundamental relation $dU=T\,dS-P\,dV+\mu\,dN$, and with it the entropy
$S(E,V,N)$ and energy $U(S,V,N)$ of a gas. But there is a quiet practical problem with those two
functions: *nobody controls a system's entropy or energy.* A laboratory thermostat fixes the
temperature $T$, not the entropy; a piston fixes the pressure $P$, not the volume; a particle
reservoir fixes the chemical potential $\mu$, not the number. The natural variables of $S$ and $U$
are the wrong ones. This notebook builds the functions whose natural variables *are* the ones we
hold fixed — the **thermodynamic potentials** $F(T,V,N)$, $H(S,P,N)$, $G(T,P,N)$, $\Phi(T,V,\mu)$ —
and the single mathematical machine that generates them all: the **Legendre transform**.

That machine is not new. It is exactly the operation that took the Lagrangian to the Hamiltonian
in Volume II, $H(q,p)=\max_{\dot q}[p\dot q-L(q,\dot q)]$, trading the velocity $\dot q$ for its
conjugate slope $p=\partial L/\partial\dot q$. Here it trades entropy for temperature,
$F=U-TS$, and volume for pressure, and number for chemical potential. We make the unity literal:
we will write *one* numerical convex-conjugate routine and run it on *both* the mechanical
$L=\tfrac12 m\dot q^2\to H=p^2/2m$ and the thermodynamic $U(S)\to F(T)$ — the same code, two
domains. This structural unity across the whole course is the heart of the notebook.

From the potentials, three things follow, and we develop each in full. Their **first derivatives**
return the conjugate variables ($-S=(\partial F/\partial T)_V$, and so on). Their **mixed second
partials** are equal regardless of order — Clairaut's theorem — and that single fact, applied to
the four potentials, gives the four **Maxwell relations**, a web of non-obvious identities that
trade a hard-to-measure quantity for an easy one. (The Maxwell relations get the room the subject
deserves: all four, each derived as a mixed partial, each verified, each with its physical use.)
Their **pure second derivatives** are the **response functions** — the heat capacity, the
compressibility — whose positivity is nothing other than thermodynamic stability, the convexity of
the potentials. And each potential is **minimized at equilibrium**, the thermodynamic face of the
entropy maximization of [§5.4](microstates-entropy-temperature.ipynb) — the principle that explains why things melt, mix, and order.

This notebook runs long by design; the structure it builds is the working architecture of
thermodynamics, and [§5.8](partition-function.ipynb) will ground it statistically when the canonical ensemble delivers
$F=-kT\ln Z$ directly. We work with the monatomic ideal gas in units $N=k=1$, handling the entropy
constant so that $S(U)$ and $U(S)$ are *exact* inverses — the one place a numerical Legendre
transform silently goes wrong, flagged where it matters.

> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the conjugate pairs assembling to an energy; the numerical Legendre transform reproducing
> analytic conjugates; the *same* routine giving $H=p^2/2m$ and $F=-U^*(T)$ (the sign flip); the
> potential identities; **each of the four Maxwell relations** as an equality of mixed partials;
> the free energy minimized at $m=\tanh(h/kT)$; the response functions $C_V=\tfrac32$, $\kappa_T=1/P$
> positive; and Joule's $(\partial U/\partial V)_T=0$. A ✓ is strong evidence; a ✗ is a prompt to
> *locate the discrepancy*.
>
> **Scope.** The architecture of equilibrium thermodynamics from the fundamental relation. The
> statistical grounding $F=-kT\ln Z$ and the response-functions-as-fluctuations are [§5.8](partition-function.ipynb); the grand
> potential and its ensemble later in the volume; criticality (where stability breaks) is [§5.10](ising-emergence-universality.ipynb). See
> Callen, *Thermodynamics*; Schroeder; Kardar; and Volume II (the Legendre transform $L\to H$),
> [§5.4](microstates-entropy-temperature.ipynb) (entropy maximization, the paramagnet), [§5.6](ideal-gas-fundamental-relation.ipynb) (the fundamental relation).

## Theory in brief

### The problem: the wrong natural variables

The fundamental relation makes $U$ a function of $(S,V,N)$ and $S$ a function of $(E,V,N)$. But a
laboratory does not control $S$ or $E$ — it controls $T$, $P$, $\mu$,

```{math}
:label: eq-natural-variables
\text{conjugate pairs:}\quad (S,T),\ (V,-P),\ (N,\mu),\qquad TS,\ PV,\ \mu N \text{ each an energy.}
```

We need potentials whose natural variables are the controlled ones. The Legendre transform builds
them.

### The Legendre transform

For a convex $f(x)$, the **Legendre (convex-conjugate) transform** is

```{math}
:label: eq-legendre
f^*(p)=\max_x\,[\,p\,x-f(x)\,],
```

a function of the *slope* $p=f'(x)$ rather than of $x$: it encodes $f$ by its family of tangent
lines (slope $p$, intercept $-f^*(p)$) instead of by its points, invertibly ($f^{**}=f$). This is
exactly Volume II's $H(q,p)=\max_{\dot q}[p\dot q-L]$ with $p=\partial L/\partial\dot q$. The
transform swaps a variable for its conjugate slope.

### The four thermodynamic potentials

Legendre-transforming $U(S,V,N)$ in each conjugate pair gives

```{math}
:label: eq-potentials
\begin{aligned}
F&=U-TS,&\ dF&=-S\,dT-P\,dV+\mu\,dN,&\ &(T,V,N)\\
H&=U+PV,&\ dH&=T\,dS+V\,dP+\mu\,dN,&\ &(S,P,N)\\
G&=U-TS+PV,&\ dG&=-S\,dT+V\,dP+\mu\,dN,&\ &(T,P,N)\\
\Phi&=U-TS-\mu N,&\ d\Phi&=-S\,dT-P\,dV-N\,d\mu,&\ &(T,V,\mu).
\end{aligned}
```

An honest subtlety of sign: thermodynamics *minimizes*, so the free energy is the **negative** of
the mathematician's convex conjugate, $F=-U^*(T)=\min_S[U-TS]$, not $\max$. We verify this
numerically and flag it — it is exactly where sign errors hide.

### The Maxwell relations

For a $C^2$ potential the mixed second partials commute (**Clairaut's theorem** — a calculus fact,
stated not proved). Applied to each potential, this gives one identity apiece,

```{math}
:label: eq-maxwell-relations
\Big(\tfrac{\partial T}{\partial V}\Big)_S=-\Big(\tfrac{\partial P}{\partial S}\Big)_V,\quad
\Big(\tfrac{\partial T}{\partial P}\Big)_S=\Big(\tfrac{\partial V}{\partial S}\Big)_P,\quad
\Big(\tfrac{\partial S}{\partial V}\Big)_T=\Big(\tfrac{\partial P}{\partial T}\Big)_V,\quad
\Big(\tfrac{\partial S}{\partial P}\Big)_T=-\Big(\tfrac{\partial V}{\partial T}\Big)_P,
```

from $U,H,F,G$ respectively. Their power: they trade a hard-to-measure quantity (an entropy
change) for an easy one (a pressure or volume response).

### The variational principle, and stability

Each potential is **minimized at equilibrium** under its natural constraints — $F$ at fixed
$(T,V)$, $G$ at fixed $(T,P)$ {eq}`eq-variational` — the Legendre image of "$S$ maximized at fixed
$E$" ([§5.4](microstates-entropy-temperature.ipynb)). $F=U-TS$ is the competition between energy minimization and entropy maximization, the
central idea of why matter melts, mixes, and orders. Finally, the transform flips convexity ($U$
convex in $S\Rightarrow F$ concave in $T$), and the **response functions** are the pure second
derivatives,

```{math}
:label: eq-legendre-stability
C_V=-T\Big(\tfrac{\partial^2 F}{\partial T^2}\Big)_V,\qquad
\kappa_T=-\tfrac1V\Big(\tfrac{\partial V}{\partial P}\Big)_T ,
```

whose positivity ($C_V>0$, $\kappa_T>0$) *is* thermodynamic stability. These equal the
fluctuations of [§5.8](partition-function.ipynb) ($\operatorname{Var}(E)=kT^2C_V$), and can diverge at a critical point ([§5.10](ising-emergence-universality.ipynb)).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"


def convex_conjugate(f, xgrid, p):
    """The numerical Legendre (convex-conjugate) transform f*(p) = max_x [p·x − f(x)] (eq-legendre).

    A single routine for *both* the mechanical Legendre transform (L(q̇) → H(p), Volume II)
    and the thermodynamic one (U(S) → free energy). For each slope ``p`` it maximizes p·x − f(x)
    over a grid of ``x`` (``numpy.max`` after broadcasting). Accuracy is set by the grid spacing and
    by the grid covering the maximizer x* = f'^(-1)(p).

    Parameters
    ----------
    f : callable
        The convex function, vectorized over the grid.
    xgrid : numpy.ndarray
        Grid of the original variable x (fine, and wide enough to contain the maximizer).
    p : float or numpy.ndarray
        The conjugate slope(s) at which to evaluate f*.

    Returns
    -------
    numpy.ndarray
        f*(p) at each requested slope.
    """
    p = np.atleast_1d(np.asarray(p, dtype=float))[:, None]
    return np.max(p * xgrid[None, :] - f(xgrid)[None, :], axis=1)


# --- the monatomic ideal gas (N = k = 1), with a consistent entropy constant ---
# S(U,V) = ln V + (3/2) ln U  and  U(S,V) = V^(-2/3) exp(2S/3) are EXACT inverses (constant s0 = 0).
def U_SV(S, V):
    """Energy U(S,V) = V^(-2/3) e^(2S/3) of the ideal gas — the exact inverse of S(U,V) = ln V + (3/2) ln U.

    Keeping the entropy constant fixed (here s0 = 0) on both sides is what makes S(U) and U(S)
    exact inverses; a mismatched constant is where a numerical Legendre transform silently fails.

    Parameters
    ----------
    S, V : float or numpy.ndarray
        Entropy and volume.

    Returns
    -------
    float or numpy.ndarray
        The energy U.
    """
    return V ** (-2.0 / 3.0) * np.exp(2.0 * S / 3.0)


def S_UV(U, V):
    """Entropy S(U,V) = ln V + (3/2) ln U of the ideal gas (the inverse of :func:`U_SV`)."""
    return np.log(V) + 1.5 * np.log(U)


# potentials and state functions in their natural variables (ideal gas, N = k = 1)
def F_TV(T, V):
    """Helmholtz free energy F(T,V) = U − TS = (3/2)T − T(ln V + (3/2) ln((3/2)T))."""
    return 1.5 * T - T * (np.log(V) + 1.5 * np.log(1.5 * T))


def S_TV(T, V):
    """Entropy in (T,V): S = ln V + (3/2) ln((3/2)T)."""
    return np.log(V) + 1.5 * np.log(1.5 * T)


def P_TV(T, V):
    """Pressure in (T,V): P = T/V (the ideal gas law)."""
    return T / V


def V_TP(T, P):
    """Volume in (T,P): V = T/P."""
    return T / P


def S_TP(T, P):
    """Entropy in (T,P): S = ln(T/P) + (3/2) ln((3/2)T)."""
    return np.log(T / P) + 1.5 * np.log(1.5 * T)


def T_SV(S, V):
    """Temperature in (S,V): T = (∂U/∂S)_V = (2/3)U."""
    return (2.0 / 3.0) * U_SV(S, V)


def P_SV(S, V):
    """Pressure in (S,V): P = −(∂U/∂V)_S = (2/3) V^(-5/3) e^(2S/3)."""
    return (2.0 / 3.0) * V ** (-5.0 / 3.0) * np.exp(2.0 * S / 3.0)


def V_SP(S, P):
    """Volume in (S,P), from inverting P = −(∂U/∂V)_S: V = (2/(3P))^(3/5) e^(2S/5)."""
    return (2.0 / (3.0 * P)) ** 0.6 * np.exp(2.0 * S / 5.0)


def T_SP(S, P):
    """Temperature in (S,P): T = (2/3) V(S,P)^(-2/3) e^(2S/3) (used for the adiabatic Maxwell relation)."""
    return (2.0 / 3.0) * V_SP(S, P) ** (-2.0 / 3.0) * np.exp(2.0 * S / 3.0)


def partial(f, i, point, h=1e-5):
    """A central-difference partial derivative ∂f/∂x_i at a point.

    Differentiates ``f(*point)`` with respect to argument ``i`` by the symmetric stencil
    (f(x_i + h) − f(x_i − h))/2h — the explicit finite-difference operator used throughout for the
    thermodynamic partials (and, applied twice, for the mixed partials of the Maxwell relations).

    Parameters
    ----------
    f : callable
        Function of several scalar arguments.
    i : int
        Index of the argument to differentiate.
    point : sequence of float
        The point at which to evaluate the derivative.
    h : float, optional
        Step size (default ``1e-5``).

    Returns
    -------
    float
        The partial derivative.
    """
    hi = list(point)
    lo = list(point)
    hi[i] += h
    lo[i] -= h
    return (f(*hi) - f(*lo)) / (2 * h)

## Exercise 1 — The wrong natural variables

We begin with the motivation, because it is the reason the rest of the notebook exists. The
fundamental relation $dU=T\,dS-P\,dV+\mu\,dN$ presents the energy as $U(S,V,N)$ and the entropy as
$S(E,V,N)$. Mathematically those are complete descriptions. Experimentally they are awkward,
because their independent variables — entropy, energy, particle number, volume — are mostly *not*
what an apparatus holds fixed ({numref}`fig-pl-pairs`). A thermostat sets the temperature; a
piston open to the atmosphere sets the pressure; a membrane to a reservoir sets the chemical
potential. Notice the structure: the relation pairs each "extensive" variable we struggle to
control ($S$, $V$, $N$) with an "intensive" conjugate slope we *can* control ($T$, $-P$, $\mu$),
and each pair multiplies to an energy ($TS$, $PV$, $\mu N$). The plan of the whole subject is to
trade each inconvenient variable for its convenient conjugate — and the Euler relation
$U=TS-PV+\mu N$ is the promise that nothing is lost in the trade: the conjugate products
reassemble the energy exactly.

**This exercise (worked).** Tabulate the conjugate pairs and which variables each potential will
hold fixed, then confirm the **Euler relation** $U=TS-PV+\mu N$ for the ideal gas with a chemical
potential obtained *independently* — as $\mu=-T\,\partial S/\partial N$ from the extensive
entropy $S(U,V,N)=N[\ln(V/N)+\tfrac32\ln(U/N)]$, by a finite difference in $N$ (the route of [§5.6](ideal-gas-fundamental-relation.ipynb)) —
so the relation genuinely tests the extensivity of $S$ rather than restating the definition of $G$.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    euler,
    U0,
    "Euler: U = TS − PV + μN, with μ from the independent −T ∂S/∂N route — extensivity, tested",
    rtol=1e-6,
)
validate.close(
    mu0,
    G0,
    "the independent μ equals G/N — the Gibbs potential is the chemical potential per particle",
    rtol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — The Legendre transform, geometrically

Before applying it, we build the Legendre transform from scratch and *see* what it does. The
definition $f^*(p)=\max_x[p\,x-f(x)]$ {eq}`eq-legendre` looks abstract, but it has a vivid
geometric meaning ({numref}`fig-pl-tangent`). For a fixed slope $p$, the line $y=p\,x$ sits above
the convex curve $f$ in some places and below in others; $p\,x-f(x)$ is the vertical gap, and its
maximum over $x$ occurs where the line is *parallel* to the curve, i.e. where $f'(x)=p$. At that
point the tangent line of slope $p$ touches the curve, and its $y$-intercept is exactly
$-f^*(p)$. So the transform re-describes the curve by its family of tangent lines — slope and
intercept — instead of by its points. No information is lost: for a convex function the transform
is its own inverse, $f^{**}=f$. The whole of the next several exercises is this one operation,
applied where the "curve" is a Lagrangian or an energy.

**This exercise (worked).** Implement the numerical `convex_conjugate` (a maximization over a grid)
and check it against the analytic conjugate of the test function $f(x)=x^2$, whose conjugate is
$f^*(p)=p^2/4$. The animation rolls a tangent line of slope $p$ along the curve, its intercept
tracing $-f^*(p)$.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    conj_numeric,
    conj_analytic,
    "the numerical Legendre transform reproduces the analytic convex conjugate (f=x² → f*=p²/4)",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — The Legendre transform is one operation: Volume II meets thermodynamics

This is the heart of the notebook, and the moment the course's structure shows itself. The
Legendre transform we just built is *the same operation* that took the Lagrangian to the
Hamiltonian in Volume II. There, $H(q,p)=\max_{\dot q}[p\dot q-L(q,\dot q)]$ traded the velocity
$\dot q$ for the momentum $p=\partial L/\partial\dot q$. Here, trading the entropy $S$ for the
temperature $T$ turns the energy $U(S,V)$ into the Helmholtz free energy. To prove the unity is
real and not a slogan, we run our *one* `convex_conjugate` routine on both: the mechanical
$L=\tfrac12 m\dot q^2$, which must give $H=p^2/2m$, and the thermodynamic $U(S,V)$ at fixed $V$,
which must give the free energy. One catch, and it is the honest subtlety of the whole subject:
thermodynamics *minimizes* the potential, so the free energy is the **negative** of the
mathematician's convex conjugate, $F=-U^*(T)=\min_S[U-TS]$ rather than $\max$. We verify the sign
flip explicitly, because it is exactly where errors hide.

**This exercise (worked).** Run `convex_conjugate` on $L=\tfrac12 m\dot q^2$ to recover $H=p^2/2m$
(Volume II), and on $U(S,V)$ to recover the free energy as $F=-U^*(T)$ — the same routine, two
domains, with the thermodynamic sign flip made explicit.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    H_numeric,
    p_mech**2 / (2 * m_mech),
    "the same Legendre routine sends L = ½mq̇² to H = p²/2m (Volume II)",
    rtol=1e-3,
)
validate.close(
    F_from_legendre,
    F_closed,
    "and sends U(S) to the free energy F = −U*(T) — the thermodynamic sign flip",
    rtol=2e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The four potentials and their differentials

With the machine established, we lay out its four products and check that they behave. From
$U(S,V,N)$ we built $F=U-TS$, $H=U+PV$, $G=U-TS+PV$, and $\Phi=U-TS-\mu N$ {eq}`eq-potentials`,
and each came with a differential whose structure tells us its natural variables: $dF=-S\,dT-
P\,dV+\mu\,dN$ has natural variables $(T,V,N)$, and so on. Two consistency checks confirm we have
them right. First, the potentials satisfy the obvious algebraic identities among themselves, e.g.
$G=F+PV=H-TS$. Second, and more usefully, each potential's first derivatives *return the conjugate
variables*: from $dF=-S\,dT-P\,dV$ we must have $-S=(\partial F/\partial T)_V$ and
$-P=(\partial F/\partial V)_T$. That second check is the working definition of a potential — it is
how one extracts $S$ and $P$ from $F$ — and we verify it numerically.

**This exercise (worked).** For the ideal gas, compute $F$, $G$, and $H$, confirm $G=F+PV$, and
confirm the first-derivative identities $-S=(\partial F/\partial T)_V$ and $V=(\partial
G/\partial P)_T$ (finite differences with the `partial` stencil).

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    [G_from_F, S_from_F, V_from_G],
    [G_direct, S0, V0],
    "the potentials satisfy G = F + PV and their first derivatives recover the conjugate variables (−∂F/∂T = S, ∂G/∂P = V)",
    rtol=1e-3,
)

## Exercise 5 — Maxwell relation I, from $F(T,V)$: the workhorse

Now the part the structure was built for, and the part we develop most carefully. Every potential
is a smooth function of two (or more) variables, and for any such $C^2$ function the mixed second
partial derivatives are equal regardless of the order of differentiation — **Clairaut's theorem**.
We take that as the calculus fact it is and do not prove it; what matters is what it *says* about
thermodynamics. Take $F(T,V)$ with $dF=-S\,dT-P\,dV$, so that $(\partial F/\partial T)_V=-S$ and
$(\partial F/\partial V)_T=-P$. Differentiate the first by $V$ and the second by $T$: both equal
$\partial^2 F/\partial T\,\partial V$, so $-(\partial S/\partial V)_T=-(\partial P/\partial T)_V$,
i.e.
$$\left(\frac{\partial S}{\partial V}\right)_T=\left(\frac{\partial P}{\partial T}\right)_V$$
{eq}`eq-maxwell-relations`. This is the workhorse Maxwell relation, and its power is practical: the
left side is an *entropy* change with volume — almost impossible to measure directly — while the
right side is how pressure responds to temperature at fixed volume, an easy gauge reading. The
relation lets one stand in for the other. For the ideal gas both sides equal $N/V$.

**This exercise (worked).** Verify $(\partial S/\partial V)_T=(\partial P/\partial T)_V$
numerically for the ideal gas (each via the `partial` stencil on $S(T,V)$ and $P(T,V)$), and read
off that both equal $N/V$.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    dS_dV_T, dP_dT_V, "Maxwell relation from F: (∂S/∂V)_T = (∂P/∂T)_V", rtol=1e-3
)

## Exercise 6 — Maxwell relation II, from $G(T,P)$

The same move on the next potential gives the next relation. Take $G(T,P)$ with $dG=-S\,dT+V\,dP$,
so $(\partial G/\partial T)_P=-S$ and $(\partial G/\partial P)_T=V$. Equating the mixed partial
$\partial^2 G/\partial T\,\partial P$ both ways gives $-(\partial S/\partial P)_T=(\partial
V/\partial T)_P$, that is
$$\left(\frac{\partial S}{\partial P}\right)_T=-\left(\frac{\partial V}{\partial T}\right)_P$$
{eq}`eq-maxwell-relations`. Note the **sign**: it is set entirely by the $+V\,dP$ in $dG$ (versus
the $-P\,dV$ in $dF$), and getting it from the differential rather than from memory is the whole
point of the method. Physically this trades the entropy's response to pressure (hard) for the
thermal expansion (easy — watch the volume grow as you warm it). For the ideal gas both sides are
$-N/P$.

**This exercise (worked).** Verify $(\partial S/\partial P)_T=-(\partial V/\partial T)_P$
numerically for the ideal gas, and note the sign comes from the $+V\,dP$ term in $dG$.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    dS_dP_T, -dV_dT_P, "Maxwell relation from G: (∂S/∂P)_T = −(∂V/∂T)_P", rtol=1e-3
)

## Exercise 7 — Maxwell relations III and IV, from $U(S,V)$ and $H(S,P)$: the adiabatic pair

The remaining two relations come from the potentials whose natural variables include the entropy,
$U(S,V)$ and $H(S,P)$, so they hold $S$ fixed — they are the *adiabatic* pair, statements about
processes at constant entropy. From $dU=T\,dS-P\,dV$, the mixed partial $\partial^2 U/\partial
S\,\partial V$ gives $(\partial T/\partial V)_S=-(\partial P/\partial S)_V$; from $dH=T\,dS+V\,dP$,
$\partial^2 H/\partial S\,\partial P$ gives $(\partial T/\partial P)_S=(\partial V/\partial S)_P$
{eq}`eq-maxwell-relations`. Because these hold the entropy fixed, we verify them along the
ideal-gas adiabat (where $TV^{2/3}$ and $TP^{-2/5}$ are constant), differentiating the
constant-entropy functions $T(S,V)$, $P(S,V)$, $T(S,P)$, $V(S,P)$. That completes the set: **four
potentials, four Maxwell relations**, each the equality of one potential's mixed second partials,
each with its own sign and its own use. We collect them in {numref}`fig-pl-maxwell`.

**This exercise (worked).** Verify the two adiabatic Maxwell relations numerically — $(\partial
T/\partial V)_S=-(\partial P/\partial S)_V$ from $U$, and $(\partial T/\partial P)_S=(\partial
V/\partial S)_P$ from $H$ — and assemble the full table of four.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    [dT_dV_S, dT_dP_S],
    [-dP_dS_V, dV_dS_P],
    "Maxwell relations from U and H (the adiabatic pair): (∂T/∂V)_S = −(∂P/∂S)_V and (∂T/∂P)_S = (∂V/∂S)_P",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — The variational principle: free energy minimized

We now reach the physical payoff that makes the free energy the most-used quantity in physical
chemistry and condensed matter. The entropy-maximization principle of [§5.4](microstates-entropy-temperature.ipynb) says an isolated system
at fixed energy drifts to maximum entropy. Legendre-transform that statement — trade fixed energy
for fixed temperature — and it becomes: a system at fixed $(T,V)$ drifts to **minimum Helmholtz
free energy** {eq}`eq-variational`. The reason $F=U-TS$ is so powerful is written in its two terms:
minimizing $F$ balances lowering the energy $U$ against raising the entropy $S$, weighted by
temperature. At low $T$ energy wins (systems order); at high $T$ entropy wins (systems disorder).
That single competition is why ice melts, why oil and water unmix, why magnets align. We
demonstrate it on the paramagnet of [§5.4](microstates-entropy-temperature.ipynb): write its free energy per spin $F(m)=-h\,m-T\,s(m)$ with
the spin entropy $s(m)$, minimize over the magnetization $m$, and recover $m=\tanh(h/kT)$
({numref}`fig-pl-paramagnet`). Adding an interaction — an internal field proportional to $m$ — will
make this *same* minimization produce spontaneous magnetization with no external field at all, the
story of [§5.10](ising-emergence-universality.ipynb).

**This exercise (worked).** Minimize the paramagnet free energy $F(m)=-h\,m-T\,s(m)$ over $m$
(`numpy.argmin` on a grid) at several fields and temperatures, and confirm the minimizer is
$m=\tanh(h/kT)$.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    m_star_main,
    np.tanh(0.5 / 1.0),
    "minimizing F(m) over the magnetization gives m = tanh(h/kT) — equilibrium minimizes the free energy",
    atol=2e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 9 — Convexity, response functions, and stability

The second derivatives of the potentials carry the last piece of the structure, and it ties
geometry to measurable physics. The Legendre transform *flips convexity*: $U$ is convex in $S$
(its graph curves upward), and its conjugate $F$ is concave in $T$ (curves downward) — we can see
it directly in the curvatures. Those curvatures are the **response functions**. The heat capacity
is $C_V=-T(\partial^2 F/\partial T^2)_V$, and the isothermal compressibility is $\kappa_T=
-\tfrac1V(\partial V/\partial P)_T$ {eq}`eq-legendre-stability`. Here is the deep point: their **positivity
is thermodynamic stability**. $C_V>0$ means adding heat raises the temperature (a system that
cooled as you heated it would run away); $\kappa_T>0$ means squeezing raises the pressure. Each is
exactly the statement that the relevant potential is convex/concave the right way — stability *is*
convexity. For the ideal gas $C_V=\tfrac32$ and $\kappa_T=1/P$, both safely positive. These same
response functions will reappear in [§5.8](partition-function.ipynb) as the *fluctuations* of the canonical ensemble
($\operatorname{Var}(E)=kT^2C_V$), and at a critical point ([§5.10](ising-emergence-universality.ipynb)) they diverge as stability fails.

**This exercise (worked).** Confirm the convexity flip ($U''(S)>0$, $F''(T)<0$); then compute the
response functions as second derivatives — $C_V=-T(\partial^2 F/\partial T^2)_V$ and $\kappa_T$ —
and confirm both are positive, $C_V=\tfrac32$ and $\kappa_T=1/P$.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.check(
    d2U_dS2 > 0 and d2F_dT2 < 0,
    "the Legendre transform flips convexity: U convex in S, F concave in T",
)
validate.close(
    [C_V, kappa_T],
    [1.5, 1 / P0],
    "the response functions C_V = −T∂²F/∂T² and κ_T are second derivatives of the potentials; their positivity is stability",
    rtol=1e-2,
)

## Exercise 10 — A thermodynamic identity by a Maxwell relation

To show the Maxwell relations are tools and not ornaments, we use one to prove a result that is far
from obvious by inspection: that the internal energy of an ideal gas does not depend on its volume
at fixed temperature, $(\partial U/\partial V)_T=0$. This is **Joule's** experimental finding (a
gas expanding into vacuum at fixed temperature neither heats nor cools), and it follows in two
lines. Starting from $dU=T\,dS-P\,dV$ and dividing by $dV$ at fixed $T$, $(\partial U/\partial
V)_T=T(\partial S/\partial V)_T-P$. Now apply the $F$-Maxwell relation of Exercise 5,
$(\partial S/\partial V)_T=(\partial P/\partial T)_V$, to convert the unmeasurable entropy term
into a pressure response: $(\partial U/\partial V)_T=T(\partial P/\partial T)_V-P$. For the ideal
gas $P=NkT/V$ gives $T(\partial P/\partial T)_V=T\cdot N k/V=P$, so the two terms cancel exactly and
$(\partial U/\partial V)_T=0$. The energy depends only on temperature — equipartition again, now
*derived* from a Maxwell relation.

**This exercise (student).** Use the $F$-Maxwell relation to evaluate $(\partial U/\partial V)_T=
T(\partial P/\partial T)_V-P$ for the ideal gas, and confirm it vanishes (Joule's result).

In [ ]:
# (solution hidden on the public site)


### Validation 10

In [ ]:
validate.close(
    dU_dV_T,
    0.0,
    "for an ideal gas (∂U/∂V)_T = 0 (Joule), shown via the F-Maxwell relation",
    atol=1e-3,
)

## Exercise 11 — The architecture of thermodynamics

Step back and see the building whole. We started from a single differential, the fundamental
relation $dU=T\,dS-P\,dV+\mu\,dN$ of [§5.6](ideal-gas-fundamental-relation.ipynb), and one operation, the **Legendre transform** of Volume
II. From them the entire structure of equilibrium thermodynamics assembled itself. The transform
generated the four **potentials**, each tuned to the variables a laboratory actually controls.
Their **first derivatives** return the conjugate variables. Their **mixed second partials**, equal
by Clairaut, are the four **Maxwell relations** — the web of identities that trade hard
measurements for easy ones, which we built one by one. Their **pure second derivatives** are the
**response functions**, whose positivity is **stability**, the convexity of the potentials made
physical. And each potential is **minimized at equilibrium**, the Legendre image of entropy
maximization, the principle behind why matter melts, mixes, and orders. Thermodynamics is not a
pile of formulas to memorize; it is one architecture — one transform, four potentials, four
relations, one variational principle — hanging from $dU=T\,dS-P\,dV+\mu\,dN$.

**This exercise (synthesis).** No new computation: the structure is the result. The same machine
that turned a Lagrangian into a Hamiltonian turned an energy into a family of free energies, and in
doing so organized all of equilibrium thermodynamics. The next notebook ([§5.8](partition-function.ipynb)) grounds it in
statistics — the canonical ensemble will hand us $F=-kT\ln Z$ directly, and the response functions
we computed here will turn out to *be* the fluctuations of that ensemble.

## Notebook summary

From the fundamental relation of [§5.6](ideal-gas-fundamental-relation.ipynb) and the Legendre transform of Volume II, this notebook built
the working architecture of thermodynamics.

- **The wrong natural variables** {eq}`eq-natural-variables`: $U(S,V,N)$ and $S(E,V,N)$ depend on
  uncontrolled variables; the conjugate pairs $(S,T),(V,-P),(N,\mu)$ each carry energy and rebuild
  $U=TS-PV+\mu N$ (Euler).
- **The Legendre transform** {eq}`eq-legendre`: $f^*(p)=\max_x[px-f(x)]$, the tangent-line
  re-encoding; our one `convex_conjugate` routine reproduced $f^*=p^2/4$, sent $L=\tfrac12 m\dot
  q^2\to H=p^2/2m$ (Volume II), and sent $U(S)\to F=-U^*(T)$ — the thermodynamic **sign flip**
  (minimize, not maximize).
- **The four potentials** {eq}`eq-potentials`: $F,H,G,\Phi$ with their differentials and natural
  variables; first derivatives recover the conjugate variables ($-S=(\partial F/\partial T)_V$).
- **The four Maxwell relations** {eq}`eq-maxwell-relations`: one per potential, each the equality
  of mixed partials (Clairaut), each verified to six digits — the isothermal pair from $F,G$ and
  the adiabatic pair from $U,H$, each trading a hard measurement for an easy one.
- **The variational principle** {eq}`eq-variational`: equilibrium minimizes $F$ at fixed $(T,V)$;
  the paramagnet's $F(m)$ is minimized at $m=\tanh(h/kT)$ — the energy-versus-entropy competition.
- **Stability** {eq}`eq-legendre-stability`: the transform flips convexity; the response functions
  $C_V=\tfrac32$ and $\kappa_T=1/P$ are positive second derivatives — stability *is* convexity. A
  Maxwell relation even gives Joule's $(\partial U/\partial V)_T=0$.

One transform, four potentials, four relations, one variational principle — the whole of
equilibrium thermodynamics, hanging from $dU=T\,dS-P\,dV+\mu\,dN$.

## Outlook

- **The canonical ensemble ([§5.8](partition-function.ipynb)).** The partition function delivers $F=-kT\ln Z$ directly,
  grounding the Helmholtz potential statistically — and the response functions of this notebook
  turn out to be **fluctuations** ($\operatorname{Var}(E)=kT^2C_V$).
- **The grand potential and its ensemble.** $\Phi(T,V,\mu)$ comes into its own when particle number
  fluctuates — the grand canonical ensemble, later in the volume.
- **Stability and its breakdown ([§5.10](ising-emergence-universality.ipynb)).** At a critical point the response functions diverge and the
  convexity of the potentials fails; the same $F$-minimization, with an interaction added, produces
  spontaneous order.
- **Phase equilibria.** The Clausius–Clapeyron relation is a Maxwell-relation / Gibbs-potential
  application — coexistence curves from the equality of chemical potentials.
- **Cross-reference** Volume II (the Legendre transform $L\to H$), [§5.4](microstates-entropy-temperature.ipynb) (entropy maximization, the
  paramagnet), [§5.6](ideal-gas-fundamental-relation.ipynb) (the fundamental relation).

In [ ]:
from ecp.style import footer

footer()